# Train the Aurora wake word model (Colab)

This notebook mirrors `wake_word/scripts/train.py`. It clones the repo, installs the
training dependencies, lets you upload your `wake_word/data` folder as a zip, trains
the CNN and exports `aurora.onnx` + `aurora.json` for download.

It also runs locally (e.g. headless via papermill) for testing: the Colab-only
steps (clone / upload / download) are skipped automatically.

**Workflow:** Runtime -> Run all, upload `data.zip` when prompted, then download the
two model files and drop them into `wake_word/models/` in your checkout.

In [ ]:
# --- Parameters (papermill can override these) ---
REPO_URL = "https://github.com/abfo/aurora"  # only used on Colab
DATA_DIR = "wake_word/data"
MODEL_OUT_DIR = "wake_word/models"
EPOCHS = 40
AUGMENT_FACTOR = 6
MAX_HARD_NEGATIVE_RATE = 0.05  # cap on 'Alexa' trigger rate; recall is maximized subject to this

In [ ]:
import os, sys, subprocess

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.isdir("aurora"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("aurora")
else:
    # Walk up to the repo root (the directory that contains wake_word/).
    while not os.path.isdir("wake_word") and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir("..")

print("cwd:", os.getcwd(), "| IN_COLAB:", IN_COLAB)
assert os.path.isdir("wake_word"), "Could not locate the wake_word package"

In [ ]:
# Install training dependencies (Colab). Locally we assume they are installed.
if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "wake_word/requirements-train.txt"],
        check=True,
    )

## Upload your data

On Colab you'll be prompted to upload a `data.zip` containing the `positives/`,
`hard_negatives/`, `negatives/` (and optional `background/`) folders. Create it from
your checkout, e.g. on the PC: zip the contents of `wake_word/data`.

In [ ]:
if IN_COLAB:
    import zipfile
    from google.colab import files
    os.makedirs(DATA_DIR, exist_ok=True)
    print("Upload data.zip (a zip of your wake_word/data folder)...")
    uploaded = files.upload()
    for fname in uploaded:
        with zipfile.ZipFile(fname) as zf:
            zf.extractall(DATA_DIR)

assert os.path.isdir(DATA_DIR), f"No data found at {DATA_DIR}"
print("Data dir:", DATA_DIR, "->", os.listdir(DATA_DIR))

In [ ]:
# Train (reuses the exact same code as the local CLI).
subprocess.run([
    sys.executable, "wake_word/scripts/train.py",
    "--data-dir", DATA_DIR,
    "--out-dir", MODEL_OUT_DIR,
    "--epochs", str(EPOCHS),
    "--augment-factor", str(AUGMENT_FACTOR),
    "--max-hard-negative-rate", str(MAX_HARD_NEGATIVE_RATE),
], check=True)

In [ ]:
# Evaluate (recall / false-alarm sweep + Alexa rejection).
subprocess.run([
    sys.executable, "wake_word/scripts/evaluate.py",
    "--data-dir", DATA_DIR,
    "--model", os.path.join(MODEL_OUT_DIR, "aurora.onnx"),
], check=True)

In [ ]:
# Download the trained model (Colab). Locally the files are already on disk.
if IN_COLAB:
    from google.colab import files
    files.download(os.path.join(MODEL_OUT_DIR, "aurora.onnx"))
    files.download(os.path.join(MODEL_OUT_DIR, "aurora.json"))
else:
    print("Wrote:", os.path.join(MODEL_OUT_DIR, "aurora.onnx"))

## Deploy

Put `aurora.onnx` and `aurora.json` into `wake_word/models/`, commit them, and
`git pull` on the Raspberry Pi. No retraining is needed on the Pi.